In [ ]:
import pandas as pd 
import re 
import unicodedata 
from pathlib import Path 

In [ ]:
# importing the datasets, setting up paths 
DATA_DIR = Path('data')
INPUT_PATH = DATA_DIR / 'filings_labeled.csv'
OUTPUT_PATH = DATA_DIR / 'filings_clean.csv'

df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} filings")
print(f"Columns: {list(df.columns)}")

In [ ]:
# merging the datasets 
df_text = pd.read_csv(DATA_DIR/ "filings_with_text.csv")[['accessionNumber', 'bodyText', 'exhibitUrl']]

df = df.merge(df_text, on='accessionNumber', how='left')

missing_text = df['bodyText'].isna().sum()
print(f"Filings with text    : {df['bodyText'].notna().sum()}")
print(f"Filings missing text : {missing_text}")
print(f"Total filings        : {len(df)}")

In [ ]:
# Layer 1: Zero-width unicode characters 

def remove_zero_width_chars(text: str) -> str:
    """
    Removes zero-width and invisible unicode characters embedded
    during HTML rendering that break downstream regex matching.
    Zero-width chars are removed entirely; non-breaking spaces
    are replaced with regular spaces to preserve word boundaries.
    """
    # Remove entirely — these have no spacing function
    zero_width = ['\u200b', '\u200c', '\u200d', '\u200e', '\u200f', '\u00ad', '\ufeff']
    for char in zero_width:
        text = text.replace(char, '')

    # Replace with regular space — these act as spacing characters
    text = text.replace('\xa0', ' ')

    return text


# Apply Layer 1 to dataset
df['cleanText'] = df['bodyText'].apply(remove_zero_width_chars)

changed = (df['cleanText'] != df['bodyText']).sum()
print(f"Layer 1 applied.")
print(f"Filings affected : {changed}")
print(f"Total filings    : {len(df)}")


In [ ]:
# Spot check Layer 1 — look at a filing we know was affected
row = df[df['ticker'] == 'APH'].iloc[0]
print("First 300 chars of cleanText:")
print(row['cleanText'][:300])

In [ ]:
def remove_sgml_headers(text: str) -> str:
    # Pass 1: remove full SGML line with filename
    text = re.sub(
        r'^EX-99[\.\d]*\s+\d+\s+\S+\.(htm|html|txt)\s+',
        '', text, flags=re.IGNORECASE
    )
    # Pass 1b: catch underscore variants e.g. "_1 2 filename.htm EX-99_1"
    text = re.sub(
        r'^[\w\-]*\s+\d+\s+\S+\.htm\s+EX-99[\._][\d\.]*\s*',
        '', text, flags=re.IGNORECASE
    )
    # Pass 2: remove repeated EX-99 type labels
    text = re.sub(
        r'^(EX-99[\.\d]*\s*)+(Document\s*)?',
        '', text, flags=re.IGNORECASE
    )
    # Pass 3: remove EXHIBIT/Exhibit 99.x labels — loop until none remain
    changed = True
    while changed:
        new = re.sub(
            r'^EXHIBIT\s+99[\.\d]*\s*(Document)?\s*',
            '', text, flags=re.IGNORECASE
        )
        new = re.sub(
            r'^Exhibit\s+99[\.\d]*\s*(Document)?\s*',
            '', new, flags=re.IGNORECASE
        )
        changed = new != text
        text = new
    # Pass 4: catch filenames like "ex_770468.htm Exhibit 99.1"
    text = re.sub(
        r'^\S+\.(htm|html|txt)\s+Exhibit\s+99[\.\d]*\s*(Document)?\s*',
        '', text, flags=re.IGNORECASE
    )
    # Pass 5: catch orphaned "Document" word
    text = re.sub(r'\bDocument\b\s*', '', text, count=1, flags=re.IGNORECASE)
    # Pass 6: catch FICO-style prefixes
    text = re.sub(
        r'^[A-Z0-9\s\-]+(?:EX-99[\.\d]*|8-K)?\s*Exhibit\s+99[\.\d]*\s*',
        '', text, flags=re.IGNORECASE
    )
    # Pass 7 (catch-all): strip any leading preamble that ends with "Exhibit 99.x"
    # within the first 500 chars — handles patterns like "NEWS RELEASE OF … Exhibit 99"
    m = re.search(r'\bExhibit\s+99[\.\.\d]*\s*(?:Document\s*)?', text[:500], re.IGNORECASE)
    if m:
        text = text[m.end():]
    return text.strip()

# Test on the problematic FICO filing
row = df[df['ticker'] == 'FICO'].iloc[8]
result = remove_sgml_headers(row['bodyText'])
print(result[:200])

In [ ]:
def remove_spaced_text(text: str) -> str:
    """
    Collapses spaced-out uppercase text e.g. "N E W S   R E L E A S E"
    into normal words e.g. "NEWS RELEASE".
    Single spaces between letters are collapsed; multiple spaces are preserved
    as word boundaries.
    """
    def collapse(match):
        # Only collapse single spaces, preserve double/triple spaces as word breaks
        return re.sub(r'(?<=[A-Z]) (?=[A-Z])', '', match.group(0))

    # Match sequences of single uppercase letters separated by 1-3 spaces
    text = re.sub(r'(?<![A-Z])(?:[A-Z] {1,3}){2,}[A-Z](?![A-Z])', collapse, text)
    
    # Clean up any remaining multiple spaces
    text = re.sub(r' {2,}', ' ', text)
    
    return text


# applying Layer 3 to dataset
df['cleanText'] = df['cleanText'].apply(remove_spaced_text)

changed = (df['cleanText'] != df['bodyText'].apply(remove_zero_width_chars).apply(remove_sgml_headers)).sum()
print(f"Layer 3 applied.")
print(f"Filings affected : {changed}")
print(f"Total filings    : {len(df)}")

In [ ]:
# Layer 4: Physical address blocks 

def remove_address_blocks(text: str) -> str:
    """
    Removes physical address blocks appearing before the press release
    headline. Runs multiple passes to catch ZIP then phone patterns.
    """
    # Run twice — first pass catches ZIP, second catches trailing phone
    for _ in range(2):
        m = re.search(
            r'(?:\(\d{3}\)\s*\d{3}[-.\s]\d{4}'  # phone: (203) 265-8900
            r'|\d{5}(?:-\d{4})?(?=\s))',          # or ZIP: 06492
            text[:500]
        )
        if m:
            text = text[m.end():].strip()
        else:
            break

    return text


df['cleanText'] = df['cleanText'].apply(remove_address_blocks)

changed = (df['cleanText'] != df['bodyText'].apply(remove_zero_width_chars).apply(remove_sgml_headers).apply(remove_spaced_text)).sum()
print(f"Layer 4 applied.")
print(f"Filings affected : {changed}")
print(f"Total filings    : {len(df)}")

In [ ]:
df['cleanText'] = df['bodyText'].apply(remove_zero_width_chars)
df['cleanText'] = df['cleanText'].apply(remove_sgml_headers)
df['cleanText'] = df['cleanText'].apply(remove_spaced_text)
df['cleanText'] = df['cleanText'].apply(remove_address_blocks)

# Spot check VRSK
row = df[df['ticker'] == 'VRSK'].iloc[0]
print(row['cleanText'][:200])

In [ ]:
# Random spot check of Layer 4 effect
import random
random.seed(99)
sample_idx = random.sample(range(len(df)), 5)

for idx in sample_idx:
    row = df.iloc[idx]
    print(f"{'='*60}")
    print(f"Ticker: {row['ticker']}  Date: {row['filingDate']}")
    print(f"First 200 chars:")
    print(row['cleanText'][:200])
    print()

In [ ]:
# Find filings that likely have contact boilerplate
keywords = ['investor relations', 'media contact', 'for more information']

for kw in keywords:
    matches = df[df['cleanText'].str.contains(kw, case=False, na=False)]
    print(f"'{kw}' found in {len(matches)} filings")

# Show an example
example = df[df['cleanText'].str.contains('investor relations', case=False, na=False)].iloc[0]
# Find where it appears
idx = example['cleanText'].lower().find('investor relations')
print(f"\nExample — {example['ticker']} {example['filingDate']}")
print(f"Context around 'investor relations':")
print(example['cleanText'][max(0, idx-100):idx+300])

In [ ]:
# Find filings where contact keywords appear in the last 20% of the document
contact_examples = []

for idx, row in df.iterrows():
    text = row['cleanText']
    length = len(text)
    for kw in ['investor relations contact', 'media contact', 'for further information']:
        pos = text.lower().rfind(kw)
        if pos > length * 0.8:  # appears in last 20%
            contact_examples.append((row['ticker'], row['filingDate'], kw, pos, length))
            break

print(f"Filings with contact block in last 20%: {len(contact_examples)}")

# Show 3 examples
for ticker, date, kw, pos, length in contact_examples[:3]:
    row = df[df['ticker'] == ticker].iloc[0]
    print(f"\n{'='*60}")
    print(f"{ticker} {date} — '{kw}' at {pos}/{length}")
    print(row['cleanText'][pos:pos+400])

In [ ]:
def remove_contact_boilerplate(text: str) -> str:
    cutoff_patterns = [
        r'investor relations contact',
        r'media contact',
        r'press contact',
        r'for further information',
        r'for more information,?\s+contact',
        r'for more information,?\s+press',    # MSFT: 'for more information, press only'
        r'for additional information,?\s+contact',
        r'conference call',
        r'#\s*#\s*#',
        r'[\w._%+-]+@[\w.-]+\.[a-zA-Z]{2,}',
        r'\bcontact information\b',
        r'\bcontacts\b(?:\s+\w+\s+\w+\s+\w+){0,3}\s+[\+\(]?\d', # 'Contacts Jennifer +1'
        r'\bcontact:\s',
        r'\bSOURCE:\s+[A-Z]',
    ]

    half = len(text) // 2

    changed = True
    while changed:
        changed = False
        for pattern in cutoff_patterns:
            m = re.search(pattern, text[half:], re.IGNORECASE)
            if m:
                cut_pos = half + m.start()
                text = text[:cut_pos].strip()
                half = len(text) // 2
                changed = True
                break

    return text

In [ ]:
# Layer 6: About [Company] sections 

def remove_about_section(text: str) -> str:
    """
    Removes 'About [Company]' boilerplate at the end of press releases.
    
    Requires the About header to be followed by company description language
    within 300 chars — either a stock ticker or 'is a/an' opener.
    This avoids false positives where 'about' appears mid-sentence.
    Only removes if found after 50% of the document.
    """
    patterns = [
        # Original — legal suffix in name (e.g. 'About Analog Devices, Inc.')
        r'about [a-z\s,\.\-\&]+ (inc|corp|co|ltd|llc|plc|corporation|company)[.,]',
        # New — 'About Adobe' style, but confirmed by company description following it
        r'about [A-Z][a-zA-Z\s\.\,\-\&]{1,50}(?=.{0,300}(?:\((?:NYSE|Nasdaq|NASDAQ):\s*[A-Z]+\)|is a leading|is a global|is an |was founded|empowers|provides solutions|delivers ))',
    ]

    earliest_hit = None
    for pattern in patterns:
        m = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
        if m and m.start() > len(text) * 0.5:
            if earliest_hit is None or m.start() < earliest_hit:
                earliest_hit = m.start()

    if earliest_hit is not None:
        text = text[:earliest_hit].strip()

    return text

In [ ]:
# Layer 7: Whitespace normalization 

def normalise_whitespace(text: str) -> str:
    """
    Normalises all whitespace to single spaces and strips
    leading/trailing whitespace. Done last to clean up any
    extra spaces introduced by earlier cleaning layers.
    """
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


# Test
before = df['cleanText'].copy()
df['cleanText'] = df['cleanText'].apply(normalise_whitespace)
changed = (df['cleanText'] != before).sum()

print(f"Layer 7 applied.")
print(f"Filings affected : {changed}")
print(f"Total filings    : {len(df)}")

# Spot check
row = df[df['ticker'] == 'APH'].iloc[0]
print(f"\nAPH first 300 chars:")
print(row['cleanText'][:300])

In [ ]:
df['cleanText'] = df['bodyText'].apply(remove_zero_width_chars)
df['cleanText'] = df['cleanText'].apply(remove_sgml_headers)
df['cleanText'] = df['cleanText'].apply(remove_spaced_text)
df['cleanText'] = df['cleanText'].apply(remove_address_blocks)
df['cleanText'] = df['cleanText'].apply(remove_contact_boilerplate)
df['cleanText'] = df['cleanText'].apply(remove_about_section)
df['cleanText'] = df['cleanText'].apply(normalise_whitespace)
df['cleanTextLen'] = df['cleanText'].str.len()

In [ ]:
print('=' * 65)
print('FINAL DATA QUALITY SCORECARD')
print('=' * 65)

checks = {
    'SGML/Exhibit header in first 300 chars'      : df['cleanText'].str[:300].str.contains('EX-99|Exhibit 99', regex=True, na=False).sum(),
    'Zero-width chars anywhere'                   : df['cleanText'].str.contains('[\u200b\u200c\u200d\ufeff]', regex=True, na=False).sum(),
    'Email in last 500 chars'                     : df['cleanText'].str[-500:].str.contains(r'[\w._%+-]+@[\w.-]+\.[a-zA-Z]{2,}', regex=True, na=False).sum(),
    'Phone in last 500 chars'                     : df['cleanText'].str[-500:].str.contains(r'\(\d{3}\)\s*\d{3}[-.]\d{4}', regex=True, na=False).sum(),
    'Investor relations contact in last 500 chars': df['cleanText'].str[-500:].str.contains('investor relations contact', case=False, na=False).sum(),
    'About [Company] in last 500 chars'           : df['cleanText'].str[-500:].str.contains(r'about .{3,50} (?:inc|corp|co|ltd|llc|plc|corporation|company)[.,]', case=False, regex=True, na=False).sum(),
    'Texts shorter than 2,000 chars'              : (df['cleanTextLen'] < 2000).sum(),
    'Texts shorter than 5,000 chars'              : (df['cleanTextLen'] < 5000).sum(),
}

all_acceptable = True
for name, count in checks.items():
    pct = count / len(df) * 100
    if count == 0:
        status = '✓'
    elif pct < 2.0:
        status = '~'   # acceptable
    else:
        status = '✗'
        all_acceptable = False
    print(f'  {status}  {name:<50s}: {count:4d} ({pct:.1f}%)')

print()
print(f'  Total filings    : {len(df):,}')
print(f'  Unique companies : {df["ticker"].nunique():,}')
print(f'  Downside events  : {df["downside"].sum():,} ({df["downside"].mean():.1%})')
print(f'  Median text len  : {df["cleanTextLen"].median():.0f} chars')
print(f'  Min text len     : {df["cleanTextLen"].min():.0f} chars')
print()
print('VERDICT:', 'CLEAN — save and proceed' if all_acceptable else 'ISSUES REMAIN')

In [ ]:
short = df[df['cleanTextLen'] < 2000].sort_values('cleanTextLen')
for _, r in short.head(10).iterrows():
    print(f"{r['ticker']} {r['filingDate']} | len={r['cleanTextLen']} | downside={r['downside']}")
    print(r['cleanText'][:300])
    print('─' * 60)

In [ ]:
for ticker, date in [('COIN', '2024-08-01'), ('CB', '2026-02-03'), ('PRU', '2026-04-14')]:
    row = df[(df['ticker'] == ticker) & (df['filingDate'] == date)].iloc[0]
    # Find original body text
    orig = df_text[df_text['accessionNumber'] == row['accessionNumber']]['bodyText'].iloc[0]
    print(f"{ticker} {date} | original len={len(orig)} | clean len={row['cleanTextLen']}")
    print(f"Original first 300: {orig[:300]}")
    print(f"Clean text: {row['cleanText']}")
    print('─' * 60)

In [ ]:
bad_filings = []
for _, r in df[df['cleanTextLen'] < 3000].iterrows():
    text = r['cleanText']
    text_lower = text.lower()
    
    # Strong table-only signals
    table_signals = [
        len(re.findall(r'table of contents', text_lower)),           # PRU pattern
        len(re.findall(r'condensed consolidated', text_lower)),      # COIN pattern  
        len(re.findall(r'chubb limited news release', text_lower)),  # CB pattern
        len(re.findall(r'in thousands|in millions', text_lower)),
    ]
    
    # Genuine narrative signals — full sentences, not table labels
    narrative_signals = [
        bool(re.search(r'[A-Z][^.!?]{40,}[.!?]', text)),   # sentence of 40+ chars
        bool(re.search(r'\b(announced|reported|exceeded|increased|decreased|grew|declined)\b', text_lower)),
        bool(re.search(r'\b(compared to|year over year|prior year|quarter over quarter)\b', text_lower)),
    ]
    
    strong_table = any(s >= 3 for s in table_signals)
    has_narrative = sum(narrative_signals) >= 2
    
    if strong_table and not has_narrative:
        bad_filings.append({
            'accessionNumber': r['accessionNumber'],
            'ticker': r['ticker'],
            'filingDate': r['filingDate'],
            'cleanTextLen': r['cleanTextLen'],
            'preview': text[:150]
        })

bad_df = pd.DataFrame(bad_filings)
print(f"Defective exhibits: {len(bad_df)}")
if len(bad_df):
    print(bad_df[['ticker', 'filingDate', 'cleanTextLen', 'preview']].to_string())

In [ ]:
# Drop defective exhibits
bad_accessions = bad_df['accessionNumber'].tolist()
df = df[~df['accessionNumber'].isin(bad_accessions)].reset_index(drop=True)

# Final counts
df['cleanTextLen'] = df['cleanText'].str.len()

print(f"Dropped {len(bad_accessions)} defective exhibits")
print(f"Final sample: {len(df):,} filings | {df['ticker'].nunique()} companies | {df['downside'].sum()} downside ({df['downside'].mean():.1%})")
print(f"Median text length: {df['cleanTextLen'].median():.0f} chars")

# Save
cols_to_save = [
    'ticker', 'cik', 'accessionNumber', 'filingDate', 'filingDatetime',
    'event_date', 'event_date_m1', 'event_date_p1',
    'car_0_1', 'car_0_1_mktadj', 'downside',
    'cleanText', 'cleanTextLen', 'exhibitUrl'
]
df[cols_to_save].to_csv('data/filings_clean.csv', index=False)
print("Saved to data/filings_clean.csv")

In [ ]:
df_clean = pd.read_csv("data/filings_clean.csv")
df_clean = df_clean.drop(columns=["car_0_1", "car_0_1_mktadj", "downside"], errors="ignore")
df_clean = df_clean.merge(
    df[["accessionNumber", "car_0_1", "car_0_1_mktadj", "downside"]],
    on="accessionNumber", how="left"
)
df_clean.to_csv("data/filings_clean.csv", index=False)
print(f"filings_clean.csv updated: {len(df_clean)} filings")
print(f"Downside rate: {df_clean['downside'].mean():.1%}")
print(f"Downside count: {df_clean['downside'].sum()}")